# Homework12

Exercises with Neural Networks

## Goals

- Get familiar with neural network setup, design, data preparation and training process
- Practice setting up the ingredients and parameters for the training loop
- Experiment with neural networks for classification tasks


### Setup

Run the following 2 cells to import all necessary libraries and helpers for this homework

In [ ]:
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/data_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/image_utils.py
!wget -q https://github.com/PSAM-5020-2026S-A/5020-utils/raw/main/src/nn_utils.py
!wget -qO- https://github.com/PSAM-5020-2026S-A/5020-utils/releases/latest/download/imagenette.tar.gz | tar xz

In [ ]:
from os import listdir
from PIL import Image as PImage

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from torch import nn, Tensor
from torch.optim import Adagrad, Adam, SGD

from data_utils import classification_error, display_confusion_matrix
from image_utils import get_pixels

from nn_utils import get_labels, get_num_params

## Neural Networks and Classification

We've set up neural networks for regression and classification in the [WK11](https://github.com/PSAM-5020-2026S-A/WK11) and [WK12](https://github.com/PSAM-5020-2026S-A/WK12) notebooks.

The steps for training a neural network are similar to all of the ML processes we've seen so far:
- Data split
- Data pre-preprocessing
- Training/Fitting
- Validation

The specific steps for training a classification network are:

- Load dataset and do any kind of pre-pre-processing (like get pixels and labels for each image)
- Split data into train/test datasets
- Perform any kind of pre-processing (like scaling or encoding)
- Load features and labels into `Tensors`
- Build a NN model
- Set up an optimizer
- Pick a cost/loss function
- Implement an evaluation function and any other kind of visualization that helps quantify/evaluate the model
- Train model
- Evaluate

### Start with the Data

As always, we start with the data. 

We're using the `imagenette` datasets, which is a small subset of the [ImageNet](https://www.image-net.org/) dataset. ImageNet has millions of images and $10\text{,}000$ classes. The `imagenette` subset has about $5\text{,}000$ images that we'll classify into $10$ classes.

The dataset is already split into two directories with training and testing datasets.

The following cell creates file lists for the training and test datasets.

In [ ]:
TRAIN_DIR = "./data/image/imagenette/train"
TEST_DIR = "./data/image/imagenette/test"

train_files = sorted(fname for fname in listdir(TRAIN_DIR) if fname.endswith("jpg"))
test_files = sorted(fname for fname in listdir(TEST_DIR) if fname.endswith("jpg"))

### Extract Features

We have to go through the files and extract labels and pixels for each image.

Pixels can be extracted with the `getdata()` function of `PIL.Image` objects or the `get_pixels()` function from our `image_utils` library.

The images are in color. We can either convert them to grayscale, or extract $3$ separate images, one from each channel, and then combine them sequentially. The `img.split()` function might help with the second strategy.

The label for each file can be extracted by reading the first item after splitting the filename on `_`.

These eventually have to be encoded into whole numbers.

In [ ]:
# TODO: extract pixels
# TODO: extract labels

### Tensor It !

Once we have `pixel` and `label` lists, we can create our `Tensor` objects.

Here we're casting label `Tensor`s to `long()` to convert them to whole numbers instead of keeping them as floats.

In [ ]:
x_train = Tensor(train["pixels"])
y_train = Tensor(train["labels"]).long()

x_test = Tensor(test["pixels"])
y_test = Tensor(test["labels"]).long()

len(x_train), len(x_test), x_train.shape, x_test.shape

### Let's train !

Let's create a single layer neural network, like the first one from class, and train it with the training data.

In addition to the actual model/network, we also need an optimizer and a loss function.

In [ ]:
# TODO: Create the model and optimizer, the loss function is already defined

loss_fn = nn.CrossEntropyLoss()

### Question

<span style="color:hotpink;">
How many parameters does your model have?<br>
How many input features does the model have?<br>
How many output features?<br>
</span>

<span style="color:hotpink;">EDIT THIS CELL WITH ANSWER</span>

### The Loop

Create a training loop like we saw in class.

This loop should:
- Predict classes by feeding all of the inputs into the `model`
- Measure `loss` (this is just `loss_fn(predicted, actual)`)
- Get the optimizer to back-propagate and annotate the neurons
- Update parameters

The loop should be repeated as long as the loss keeps improving, and it doesn't look like the model is overfitting with the training data.

In order to check if the model is overfitting, we can sporadically run evaluations within the training loop in order to see if the model performs similarly with `train` and `test` data.

But ! Our network actually outputs a series of values for each image that we give it. In order to determine the exact class number of its predictions, we have to find the index of the output neuron with the largest value, which is an operation called `argmax()` (similar to `argsort()` from week 10).

It's not hard to do this manually, but we can use the `get_labels(model, inputs)` function inside the `nn_utils` file to run our `model` on all of the data in a given dataset and return the predicted labels for all of the samples.

In [ ]:
# TODO: iterate epochs
  # TODO: predict
  # TODO: measure loss
  # TODO: compute gradient and step optimizer
  # TODO: show progress

### Evaluate

This should be similar to the last error values seen during training, but sometimes it changes a bit...

Not a bad idea to check the accuracy of the model using the `classification_error()` function, and look at some confusion matrices.

In [ ]:
# TODO: classification error for train and test data
# TODO: confusion matrices for train and test data

### Interpretation

<span style="color:hotpink;">
What's going on ? Is the network learning ?
</span>

<span style="color:hotpink;">EDIT THIS CELL WITH ANSWER</span>

## Neural Networks and PCA

We are seeing how Neural Networks can be _easy_ to build and explain in generic/abstract terms (a bunch of little operators that perform weighted sums of their inputs), but in reality can be really difficult and opaque to steer.

In theory, a couple of well placed neuron layers, with the right hyperparameters, learning rate, loss function, architecture and a good amount of data, can learn to extract information like polynomial features, clusters or even PCA components. But... that's not always the case and sometimes it's not a bad idea to *encourage* the network to go down a certain path.

One way to do this is to pre-process our inputs and do a bit of feature extraction ourselves.

Let's see if we can improve this classification network by using PCA information instead of pixel data.

### Add PCA

We're going to repeat the training, but this time our data is going to be scaled and PCA'd before going into the neural network.

So, the data preparation flow should be:
- Scale data for `PCA`
- Perform `PCA`

We need one `StandardScaler()` object and one `PCA()` object.

The `train` data goes through the `fit_transform()` function of these objects, while the `test` data only goes through `transform()`.

For the `PCA`, we can aim for an explained variance of around $80\%$. This should reduce the number of features significantly to allow us to experiment with our network architecture.

We're working with a dataset that is $3\text{,}000$ rows by $49\text{,}152$ columns. Fitting `PCA` can take a minute.

In [ ]:
# TODO: Scale
# TODO: PCA
# TODO: Tensors

### Repeat

Re-create model, optimizer, loss function, then re-run the training loop and evaluate.

In [ ]:
# TODO: Model, Optimizer and Loss Function
# TODO: Training loop
# TODO: Evaluation

### Interpretation

<span style="color:hotpink;">
So... What happens ?<br>
How does training on the <code>PCA</code> data compare to training on the regular data ?

What else does <code>PCA</code> afford us in this case ? ...<br>
How does adding extra layers in the original network compare to adding extra layers in the <code>PCA</code> network?
</span>

<span style="color:hotpink;">EDIT THIS CELL WITH ANSWER</span>